# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamsherif04/flyrank-ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ai"):
        !git clone https://github.com/mariamsherif04/flyrank-ai.git
    os.chdir("flyrank-ai")

print("Working directory:", os.getcwd())
print("Data found:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Cloning into 'flyrank-ai'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 138 (delta 48), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 8.53 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Working directory: /content/flyrank-ai
Data found: True


## 1. My lane as an ML task (type)

Scoring/ranking task. A classification model outputs a decline probability, which combines with a rule-based baseline into a continuous score used to rank pages by review priority. Framed as ranking (not plain classification) because the deliverable is an ordered queue, not isolated labels.

In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Simulate the continuous score a classifier would output (probability), then rank by it —
# the deliverable is an ORDER, not a per-row label.
import numpy as np
np.random.seed(42)
df["decline_score"] = np.clip(df["is_declining_label"] * 0.6 + np.random.normal(0, 0.15, len(df)), 0, 1)
ranked = df.sort_values("decline_score", ascending=False)
print("Top 5 of the ranked review queue (by decline_score):")
print(ranked[["content_id", "decline_score", "is_declining_label"]].head())

Top 5 of the ranked review queue (by decline_score):
                 content_id  decline_score  is_declining_label
27920  content_18fff51d6cbd            1.0                   1
23189  content_35b9afe673f9            1.0                   1
27864  content_ed9c66725ca2            1.0                   1
12245  content_6c3a6dd1cf22            1.0                   1
2801   content_0c0712027f83            1.0                   1


## 2. Target or proxy

Proxy label: trend_direction == "down" — a rule computed from the current window, not a future outcome. Weakness: doesn't prove decline will continue. Stronger version (if extended to warehouse data): prior 90-day features → decline over next 30 days, a genuine future observed outcome.

In [4]:
# The label is a rule computed on the CURRENT window, not a future observed outcome
print(df[["trend_direction", "trend_pct", "is_declining_label"]].drop_duplicates())
print("\nShare labeled declining:", df["is_declining_label"].mean().round(3))
print("Weakness: this says a page IS currently trending down, not that it WILL still be")
print("declining next month — that would require a real prior->future split, which this")
print("single-snapshot CSV doesn't support.")

      trend_direction  trend_pct  is_declining_label
0                down      -41.4                   1
1                down      -57.7                   1
2                down      -60.9                   1
3              stable      -13.8                   0
4                down      -34.7                   1
...               ...        ...                 ...
29808              up      155.2                   0
29847              up       91.0                   0
29857              up      206.3                   0
29893              up      508.3                   0
29941              up       90.5                   0

[2716 rows x 3 columns]

Share labeled declining: 0.542
Weakness: this says a page IS currently trending down, not that it WILL still be
declining next month — that would require a real prior->future split, which this
single-snapshot CSV doesn't support.


## 3. Success metric

Precision@50. Reviewers act only on the top of the ranked list, so what matters is accuracy within that slice, not overall accuracy across all pages (which would be inflated by the majority non-declining class).

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

p50 = precision_at_k(df["decline_score"].values, df["is_declining_label"].values, 50)
print(f"Precision@50 on the simulated score: {p50:.3f}")
print("Chosen because reviewers only ever act on the top of the list — overall accuracy")
print("would be misleadingly high given the majority non-declining class:")
print("Majority-class accuracy baseline:", round(1 - df['is_declining_label'].mean(), 3))

Precision@50 on the simulated score: 1.000
Chosen because reviewers only ever act on the top of the list — overall accuracy
would be misleadingly high given the majority non-declining class:
Majority-class accuracy baseline: 0.458


## 4. The unit of analysis, as a real dataframe

One row = one content item. Load content_refresh_anonymized.csv, filter impressions_90d > 0 and content_age_days >= 90, dedupe by content_id. Grain confirmed via .shape and .head().

In [6]:
unit_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")
print("Shape:", unit_df.shape)
print("Duplicate content_id count (should be 0):", unit_df["content_id"].duplicated().sum())
unit_df.head()

Shape: (30000, 46)
Duplicate content_id count (should be 0): 0


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,decline_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,0.674507
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,0.579260
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,0.697153
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,0.228454
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,0.564877


## 5. Why ML beats a fixed rule here

Baseline rule precision@50 = 0.240; random forest precision@50 = 0.740. Fixed thresholds can't capture interaction effects — e.g., staleness only matters combined with demand, thinness only matters combined with visibility. Tree-based models weight these interactions; single if-statement rules cannot.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Hand rule: stale (180+ days since update) AND still visible (500+ impressions)
df["hand_rule_score"] = (
    (df["days_since_last_update"] >= 180).astype(int)
    * (df["impressions_90d"] >= 500).astype(int)
    * df["impressions_90d"]
)
hand_p50 = precision_at_k(df["hand_rule_score"].values, df["is_declining_label"].values, 50)

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(Xtr, ytr)
rf_scores_full = rf.predict_proba(X)[:, 1]
rf_p50 = precision_at_k(rf_scores_full, y.values, 50)

print(f"Baseline rule Precision@50: {hand_p50:.3f}")
print(f"Random forest Precision@50: {rf_p50:.3f}")
print(f"Improvement: {rf_p50/hand_p50:.1f}x")

Baseline rule Precision@50: 0.680
Random forest Precision@50: 1.000
Improvement: 1.5x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.